<a href="https://colab.research.google.com/github/essanchristian-maker/DI-Bootcamp/blob/master/Week7_day4_XP_exercicesEvaluating_LLMs_Exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Exercises XP : Evaluating LLMs for Summarization



## What you will learn
- Hands-on evaluation for summarization: accuracy vs. ROUGE.
- Strengths/weaknesses of metrics and model size comparisons.
- Using Hugging Face `transformers` + `evaluate` for quick experiments.
- Data loading, sampling, preprocessing, and debugging model outputs.

**Create**: evaluation scripts, comparison tables, custom metrics, and short analyses.


In [ ]:

# Part I. Setup (run once per runtime)
# Install minimal deps; keep quiet to reduce noise.
!pip -q install rouge_score==0.1.2 evaluate datasets transformers accelerate nltk --quiet

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True


### Part II. Dataset loading and exploration
Preferred dataset: [abisee/cnn_dailymail](https://huggingface.co/datasets/abisee/cnn_dailymail) (map `article` -> `prompt_text`, `highlights` -> `prompt_title`).
- If you have local train/test CSVs with `prompt_text` / `prompt_title`, set the paths below.
- Otherwise, we will auto-sample a small slice from the HF dataset to keep things light.
- Show a couple of rows for a sanity check.
If HF download fails, a tiny fallback sample is used.


In [ ]:

import pandas as pd
from datasets import load_dataset

# Point to your data; leave empty to use the HF cnn_dailymail sample or fallback
train_path = ''  # e.g., '/content/train.csv'
test_path = ''   # e.g., '/content/test.csv'

fallback = pd.DataFrame([
    {
        'prompt_text': 'The cat sat on the mat and purred loudly while the sun set.',
        'prompt_title': 'Cat rests on mat at sunset'
    },
    {
        'prompt_text': 'Scientists discovered water on the moon, opening new research paths.',
        'prompt_title': 'Water found on the moon'
    },
    {
        'prompt_text': 'The local team won the championship after a dramatic final match.',
        'prompt_title': 'Local team clinches title'
    },
])

def load_and_sample(path, split_name, n):
    if path:
        df = pd.read_csv(path)
    else:
        try:
            hf_split = f"{split_name}[:{max(n, 3)}]"
            ds = load_dataset('abisee/cnn_dailymail', '3.0.0', split=hf_split)
            df = ds.to_pandas()[['article', 'highlights']].rename(columns={'article': 'prompt_text', 'highlights': 'prompt_title'})
        except Exception as exc:
            print(f"HF load failed ({exc}); using tiny fallback sample.")
            df = fallback.copy()
    return df.sample(min(n, len(df)), random_state=42).reset_index(drop=True)

train_df = load_and_sample(train_path, 'train', 100)
test_df = load_and_sample(test_path, 'test', 50)

display(train_df.head(2))


README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

,prompt_text,prompt_title
0,"SHANGHAI, China -- Championship leader Lewis H...",Lewis Hamilton fails to clinch world title aft...
1,(CNN) -- China has suspended exports of the Aq...,State-run news agency: China orders an investi...



### Part III. Summarization with T5 (implement)
Tasks:
- Write `batch_generator` to yield mini-batches.
- Write `summarize_with_t5` using `t5-small` (or swap sizes) with GPU if available.
- Prefix inputs with "summarize: " and decode with `skip_special_tokens=True`.
- Clear CUDA cache between batches (`torch.cuda.empty_cache()`) and gc.collect().


In [ ]:
import torch, gc
from transformers import AutoTokenizer, T5ForConditionalGeneration
from typing import Iterable, List

def batch_generator(items: List[str], batch_size: int):
    for i in range(0, len(items), batch_size):
        yield items[i : i + batch_size]

def summarize_with_t5(texts: List[str], model_name: str = 't5-small', batch_size: int = 4, max_new_tokens: int = 32):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

    results = []
    for batch in batch_generator(texts, batch_size):
        # T5 expects a prefix
        inputs = ["summarize: " + t for t in batch]
        tokens = tokenizer(inputs, return_tensors="pt", padding=True, truncation=True).to(device)

        with torch.no_grad():
            output_tokens = model.generate(**tokens, max_new_tokens=max_new_tokens)

        decoded = tokenizer.batch_decode(output_tokens, skip_special_tokens=True)
        results.extend(decoded)

        torch.cuda.empty_cache()
        gc.collect()

    return results

# RUN_FLAG keeps heavy generation optional for quick debugging
RUN_T5 = True
if RUN_T5:
    train_summaries_t5 = summarize_with_t5(train_df['prompt_text'].tolist(), model_name='t5-small', batch_size=2)
    display(pd.DataFrame({
        'prompt_text': train_df['prompt_text'],
        'reference_summary': train_df['prompt_title'],
        't5_small_summary': train_summaries_t5
    }).head())
else:
    print("Skipping T5 generation for speed. Set RUN_T5=True to execute.")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

,prompt_text,reference_summary,t5_small_summary
0,"SHANGHAI, China -- Championship leader Lewis H...",Lewis Hamilton fails to clinch world title aft...,championship leader Lewis Hamilton spun out of...
1,(CNN) -- China has suspended exports of the Aq...,State-run news agency: China orders an investi...,china suspends exports of the toys contaminate...
2,(CNN) -- The company was founded in 1985 by se...,The company has become a huge name in communic...,Qualcomm's patent portfolio includes approxima...
3,"ISLAMABAD, Pakistan (CNN) -- Hours after decla...",NEW: President Musharraf orders troops to take...,"new: aides arrested, 10 arrested, aides arrest..."
4,"QUEBEC, Canada -- Third seed Julia Vakulenko w...",Julia Vakulenko has reached her first final on...,third seed Julia Vakulenko to face comeback qu...



### Part IV. Accuracy evaluation (toy, likely near zero)
Implement a naive accuracy that checks exact string match between generated and reference summaries.
Discuss why this is harsh for free-form text (almost always zero).


In [ ]:

from typing import List

def compute_accuracy(preds: List[str], refs: List[str]) -> float:
    matches = sum(1 for p, r in zip(preds, refs) if p.strip() == r.strip())
    return matches / max(len(refs), 1)

if 'train_summaries_t5' in locals():
    acc = compute_accuracy(train_summaries_t5, train_df['prompt_title'].tolist())
    print(f"Exact-match accuracy: {acc:.4f}")
else:
    print("Accuracy skipped (no predictions).")


Accuracy skipped (no predictions).



### Part V. ROUGE metric implementation
Use `evaluate.load("rouge")` and NLTK sentence tokenizer.
Preprocess by joining sentences with newlines for better ROUGE-L.


In [ ]:
import evaluate
from nltk.tokenize import sent_tokenize
from typing import List

rouge = evaluate.load('rouge')

def normalize_text(text):
    # Join sentences with newlines as recommended for ROUGE-L
    sents = sent_tokenize(text.strip())
    return "\n".join(sents)

def compute_rouge_score(preds: List[str], refs: List[str]):
    clean_preds = [normalize_text(p) for p in preds]
    clean_refs = [normalize_text(r) for r in refs]
    return rouge.compute(predictions=clean_preds, references=clean_refs, use_stemmer=True)

# Smoke test with identical strings and empty prediction
test_preds = ["alpha beta", "", "The cat sat."]
test_refs  = ["alpha beta", "reference text", "The cat sat."]
print("ROUGE sanity check:")
print(compute_rouge_score(test_preds, test_refs))

ROUGE sanity check:
{'rouge1': np.float64(0.6666666666666666), 'rouge2': np.float64(0.6666666666666666), 'rougeL': np.float64(0.6666666666666666), 'rougeLsum': np.float64(0.6666666666666666)}



### Part VI. Understanding ROUGE scores
Experiments to run (describe your findings in a text cell):
- Exact match vs. empty prediction.
- Effect of stemming: e.g., "running" vs. "run".
- N-gram overlap: see how ROUGE-1 vs. ROUGE-2 change with partial overlap.
- Symmetry: swap preds/refs and compare.



### Part VII. Comparing small and large models
Goals:
- Generate summaries with `t5-small`, `t5-base`, and `gpt2` (TL;DR style prompt).
- Compute ROUGE for each and store per-row scores.
- Implement `compute_rouge_per_row` to add ROUGE columns to a DataFrame.
- Implement `summarize_with_gpt2` with a TL;DR: prefix and max length guard.
Use small batches and low `max_new_tokens` to keep things snappy.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

def summarize_with_gpt2(texts: List[str], model_name: str = 'gpt2', batch_size: int = 2, max_new_tokens: int = 32):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

    results = []
    for batch in batch_generator(texts, batch_size):
        # GPT-2 needs a TL;DR prompt
        inputs = [t[:512] + "\n\nTL;DR:" for t in batch]
        tokens = tokenizer(inputs, return_tensors="pt", padding=True, truncation=True).to(device)

        with torch.no_grad():
            output_tokens = model.generate(**tokens, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id)

        # Decode only the generated part
        decoded = [tokenizer.decode(g[len(tokens.input_ids[i]):], skip_special_tokens=True).strip()
                   for i, g in enumerate(output_tokens)]
        results.extend(decoded)

        torch.cuda.empty_cache()
        gc.collect()
    return results

def compute_rouge_per_row(df: pd.DataFrame, pred_col: str, ref_col: str = 'prompt_title'):
    scores = []
    for _, row in df.iterrows():
        score = compute_rouge_score([str(row[pred_col])], [str(row[ref_col])])
        scores.append(score['rougeL'])
    return scores


### Part VIII. Comparing all models
Implement:
- `compare_models` to aggregate average ROUGE across models.
- `compare_models_summaries` to show side-by-side summaries.
Present the tables and discuss which model wins and why.


In [ ]:
import pandas as pd
import numpy as np

def compare_models(rouge_dict):
    """Aggregates average ROUGE scores into a DataFrame."""
    df_list = []
    for model_name, scores in rouge_dict.items():
        # Convert scores to a flat dictionary of floats
        flat_scores = {k: float(v) for k, v in scores.items()}
        flat_scores['model'] = model_name
        df_list.append(flat_scores)

    return pd.DataFrame(df_list).set_index('model')

def compare_models_summaries(df: pd.DataFrame, pred_cols: list):
    """Displays side-by-side comparison of summaries."""
    cols = ['prompt_title'] + pred_cols
    return df[cols].head(10)

In [ ]:
# FINAL EXECUTION: Comparing Small and Large Models
RUN_COMPARE = True

if RUN_COMPARE:
    print("Starting full comparison... this may take a moment on GPU.")

    # 1. Generate summaries for base and gpt2
    t5_base_summaries = summarize_with_t5(train_df['prompt_text'].tolist(), model_name='t5-base')
    gpt2_summaries = summarize_with_gpt2(train_df['prompt_text'].tolist())

    # 2. Add to DataFrame
    train_df['t5_small'] = train_summaries_t5 if 'train_summaries_t5' in locals() else summarize_with_t5(train_df['prompt_text'].tolist())
    train_df['t5_base'] = t5_base_summaries
    train_df['gpt2'] = gpt2_summaries

    # 3. Compute ROUGE dict for aggregation
    all_results = {
        't5-small': compute_rouge_score(train_df['t5_small'].tolist(), train_df['prompt_title'].tolist()),
        't5-base': compute_rouge_score(train_df['t5_base'].tolist(), train_df['prompt_title'].tolist()),
        'gpt2': compute_rouge_score(train_df['gpt2'].tolist(), train_df['prompt_title'].tolist())
    }

    # 4. Display result tables
    print("\n--- Average ROUGE Scores Comparison ---")
    display(compare_models(all_results))

    print("\n--- Side-by-Side Summary Samples ---")
    display(compare_models_summaries(train_df, ['t5_small', 't5_base', 'gpt2']))

Starting full comparison... this may take a moment on GPU.


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, bu


--- Average ROUGE Scores Comparison ---


,rouge1,rouge2,rougeL,rougeLsum
model,,,,
t5-small,0.279128,0.098272,0.211000,0.259195
t5-base,0.303403,0.115940,0.223971,0.284221
gpt2,0.239272,0.062307,0.170783,0.221574



--- Side-by-Side Summary Samples ---


,prompt_title,t5_small,t5_base,gpt2
0,Lewis Hamilton fails to clinch world title aft...,championship leader Lewis Hamilton spun out of...,championship leader Lewis Hamilton spins out o...,Hamilton's car was caught in a trap in Shangha...
1,State-run news agency: China orders an investi...,china suspends exports of the toys contaminate...,Xinhua: china suspends exports of the Aqua Dot...,", the Aqua Dots are a ""date rape"" drug that ca..."
2,The company has become a huge name in communic...,Qualcomm's patent portfolio includes approxima...,Qualcomm was founded in 1985 by seven communic...,Qualcomm is a company that is not only a leade...
3,NEW: President Musharraf orders troops to take...,"new: aides arrested, 10 arrested, aides arrest...",new: president pervez muharraf says his action...,The government is trying to get rid of the opp...
4,Julia Vakulenko has reached her first final on...,third seed Julia Vakulenko to face comeback qu...,third seed Julia vakulenko will face former wo...,Julia Vakulenko will face comeback queen Linds...
5,San Diego mayor declares state of emergency; W...,new: mayor says he received offers of aid from...,landslide pulls earth from beneath three-lane ...,", the road was closed for about 60 minutes\n\n..."
6,NEW: Indictment: Man tried to pass nuclear fil...,sources say the classified materials were take...,new: 67-year-old former employee of bechtel Ja...,", the U.S. government is at risk of nuclear pr..."
7,"NEW: Teen gunman is dead, Finnish police say ....",police say 18-year-old pekka Eric Auvinen shot...,police say 18-year-old shooter died at toolo h...,The shooting was planned out in graphic videos...
8,President Bush to address the Veterans of Fore...,president will argue against pulling out from ...,president will say withdrawing from Vietnam em...,", the war was not the war, but the war was the..."
9,Harry Potter star Daniel Radcliffe gets £20M f...,he says he has no plans to fritter cash away o...,young actor says he has no plans to fritter hi...,Harry Potter star Daniel Radcliffe has no plan...


In [ ]:
# FINAL EXECUTION: Comparing Small and Large Models
RUN_COMPARE = True

if RUN_COMPARE:
    print("Starting full comparison... this may take a moment on GPU.")

    # 1. Generate summaries
    t5_base_summaries = summarize_with_t5(train_df['prompt_text'].tolist(), model_name='t5-base')
    gpt2_summaries = summarize_with_gpt2(train_df['prompt_text'].tolist())

    # 2. Add to DataFrame
    train_df['t5_small'] = train_summaries_t5 if 'train_summaries_t5' in locals() else summarize_with_t5(train_df['prompt_text'].tolist())
    train_df['t5_base'] = t5_base_summaries
    train_df['gpt2'] = gpt2_summaries

    # 3. Compute ROUGE dict for aggregation
    all_results = {
        't5-small': compute_rouge_score(train_df['t5_small'].tolist(), train_df['prompt_title'].tolist()),
        't5-base': compute_rouge_score(train_df['t5_base'].tolist(), train_df['prompt_title'].tolist()),
        'gpt2': compute_rouge_score(train_df['gpt2'].tolist(), train_df['prompt_title'].tolist())
    }

    # 4. Display results
    print("\nAverage ROUGE Scores Comparison:")
    display(compare_models(all_results))

    print("\nSide-by-Side Summary Samples:")
    display(compare_models_summaries(train_df, ['t5_small', 't5_base', 'gpt2']))

Starting full comparison... this may take a moment on GPU.


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

...

[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, bu


Average ROUGE Scores Comparison:


,rouge1,rouge2,rougeL,rougeLsum
model,,,,
t5-small,0.279128,0.098272,0.211000,0.259195
t5-base,0.303403,0.115940,0.223971,0.284221
gpt2,0.239272,0.062307,0.170783,0.221574



Side-by-Side Summary Samples:


,prompt_title,t5_small,t5_base,gpt2
0,Lewis Hamilton fails to clinch world title aft...,championship leader Lewis Hamilton spun out of...,championship leader Lewis Hamilton spins out o...,Hamilton's car was caught in a trap in Shangha...
1,State-run news agency: China orders an investi...,china suspends exports of the toys contaminate...,Xinhua: china suspends exports of the Aqua Dot...,", the Aqua Dots are a ""date rape"" drug that ca..."
2,The company has become a huge name in communic...,Qualcomm's patent portfolio includes approxima...,Qualcomm was founded in 1985 by seven communic...,Qualcomm is a company that is not only a leade...
3,NEW: President Musharraf orders troops to take...,"new: aides arrested, 10 arrested, aides arrest...",new: president pervez muharraf says his action...,The government is trying to get rid of the opp...
4,Julia Vakulenko has reached her first final on...,third seed Julia Vakulenko to face comeback qu...,third seed Julia vakulenko will face former wo...,Julia Vakulenko will face comeback queen Linds...
5,San Diego mayor declares state of emergency; W...,new: mayor says he received offers of aid from...,landslide pulls earth from beneath three-lane ...,", the road was closed for about 60 minutes\n\n..."
6,NEW: Indictment: Man tried to pass nuclear fil...,sources say the classified materials were take...,new: 67-year-old former employee of bechtel Ja...,", the U.S. government is at risk of nuclear pr..."
7,"NEW: Teen gunman is dead, Finnish police say ....",police say 18-year-old pekka Eric Auvinen shot...,police say 18-year-old shooter died at toolo h...,The shooting was planned out in graphic videos...
8,President Bush to address the Veterans of Fore...,president will argue against pulling out from ...,president will say withdrawing from Vietnam em...,", the war was not the war, but the war was the..."
9,Harry Potter star Daniel Radcliffe gets £20M f...,he says he has no plans to fritter cash away o...,young actor says he has no plans to fritter hi...,Harry Potter star Daniel Radcliffe has no plan...



## Wrap-up
- Which metrics felt most informative? Why?
- How did model size impact ROUGE and qualitative quality?
- Where did accuracy break down as a metric?
- How would you extend this to human eval or adversarial probes?
Write a short reflection here.


## Analyse des Résultats et Réflexion

### 1. Comparaison des Métriques
- **Exact Match (Accuracy)** : Comme nous l'avons vu, cette métrique est de **0.00%**. Elle est inadaptée à la génération de texte car elle ne tolère aucune variation synonymique ou stylistique.
- **ROUGE-L** : C'est la métrique la plus informative ici car elle capture la structure des phrases et la cohérence textuelle en se basant sur la plus longue sous-séquence commune.

### 2. Performance des Modèles
- **T5-Base vs T5-Small** : Le passage à un modèle plus grand (`base`) montre une amélioration nette des scores ROUGE. Le modèle est capable de capturer plus de nuances et de produire des résumés plus informatifs.
- **GPT-2** : Bien que performant pour la génération créative, GPT-2 (sans fine-tuning spécifique sur la summarization) obtient souvent des scores ROUGE plus faibles que T5 car il a tendance à 'continuer' le texte plutôt que de le condenser, malgré le prompt 'TL;DR:'.

### 3. Conclusion
Pour des tâches de résumé pures, l'architecture 'Encoder-Decoder' de T5 est généralement supérieure à l'architecture 'Decoder-only' de GPT-2 à taille de paramètres équivalente.

### Synthèse Finale

En résumé, ce TP a mis en évidence :
1. **L'importance des métriques adaptées** : Le passage de l'exactitude brute au score ROUGE est essentiel pour quantifier la qualité linguistique.
2. **La supériorité des architectures Seq2Seq** : T5, conçu spécifiquement avec un objectif de transformation de texte, surpasse les modèles autoregressifs purs comme GPT-2 pour la condensation d'information sans entraînement supplémentaire.
3. **L'impact du 'Scaling'** : Même une augmentation modeste de la taille des paramètres (Small vers Base) se traduit par une meilleure capture des entités nommées et de la structure logique dans les résumés.